# LAV-DF Timestamp-Aware Preprocessing

Standard preprocessing samples 16 frames uniformly across the whole video.
Since LAV-DF manipulations occupy only **~9% of video duration** on average
(median fake period = 0.664s), uniform sampling yields ~1-2 fake frames out
of 16 — severely diluting the forgery signal.

**This notebook samples frames exclusively from annotated fake periods.**

| | Real videos | Fake videos |
|---|---|---|
| Standard | 16 frames uniform | 16 frames uniform (~1-2 fake) |
| **Timestamp-aware** | 16 frames uniform | **16 frames from fake period only** |

### Dataset stats
- Train+dev : 110,204 videos (balanced ~19-21k/class)
- Test       : 26,100 videos
- Fake period: min=0.164s  median=0.664s  max=1.6s
- 3 splits   : train / dev (merged) / test

In [1]:
# ── CELL 1: SETUP ─────────────────────────────────────────────────────────
!pip install librosa opencv-python-headless tqdm retina-face -q
!apt-get install -y ffmpeg -q
print('Setup done.')

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Setup done.


In [2]:
# ── CELL 2: IMPORTS & DRIVE ───────────────────────────────────────────────
import os, json, random, subprocess
import numpy as np
import cv2
import librosa
from tqdm import tqdm
from collections import Counter
from google.colab import drive
from retinaface import RetinaFace
drive.mount('/content/drive')

random.seed(42)
np.random.seed(42)
print('Imports done.')

Mounted at /content/drive
Imports done.


In [5]:
import kagglehub

path = kagglehub.dataset_download('elin75/localized-audio-visual-deepfake-dataset-lav-df')
print('Downloaded to:', path)

# Find metadata.min.json
for root, dirs, files in os.walk(path):
    for f in files:
        if 'metadata' in f.lower():
            print(os.path.join(root, f))

Using Colab cache for faster access to the 'localized-audio-visual-deepfake-dataset-lav-df' dataset.
Downloaded to: /kaggle/input/localized-audio-visual-deepfake-dataset-lav-df
/kaggle/input/localized-audio-visual-deepfake-dataset-lav-df/LAV-DF/metadata.json
/kaggle/input/localized-audio-visual-deepfake-dataset-lav-df/LAV-DF/metadata.min.json


In [8]:
# ── CELL 3: CONFIG ────────────────────────────────────────────────────────
LAV_ROOT = '/kaggle/input/localized-audio-visual-deepfake-dataset-lav-df/LAV-DF'
OUT_TRAIN = '/content/drive/MyDrive/Deepfake/lavdf_ts_processed/train'
OUT_TEST  = '/content/drive/MyDrive/Deepfake/lavdf_ts_processed/test'
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST,  exist_ok=True)

# Preprocessing params — identical to FakeAVCeleb
FRAME_SIZE    = 224
NUM_FRAMES    = 16
NUM_AUDIO_SEG = 16
SR            = 16000
N_MELS        = 128
TARGET_MEL_T  = 32

# Per-class limits
TRAIN_PER_CLASS = 200
TEST_PER_CLASS  = 80

# Min frames in fake period before falling back to uniform
MIN_FAKE_FRAMES = 4

CLASS_NAMES = {0:'Real/Real', 1:'FakeVideo/RealAudio',
               2:'RealVideo/FakeAudio', 3:'Fake/Fake'}

print('LAV_ROOT exists:', os.path.exists(LAV_ROOT))
print('OUT_TRAIN:', OUT_TRAIN)
print('OUT_TEST :', OUT_TEST)

LAV_ROOT exists: True
OUT_TRAIN: /content/drive/MyDrive/Deepfake/lavdf_ts_processed/train
OUT_TEST : /content/drive/MyDrive/Deepfake/lavdf_ts_processed/test


In [9]:
# ── CELL 4: LOAD METADATA ─────────────────────────────────────────────────
with open(os.path.join(LAV_ROOT, 'metadata.min.json')) as f:
    meta = json.load(f)

def get_label(m):
    mv = m.get('modify_video', False)
    ma = m.get('modify_audio', False)
    if   not mv and not ma: return 0
    elif     mv and not ma: return 1
    elif not mv and     ma: return 2
    else:                   return 3

for m in meta:
    m['label'] = get_label(m)

# Merge train + dev; keep test separate
train_meta = [m for m in meta if m.get('split') in ('train', 'dev')]
test_meta  = [m for m in meta if m.get('split') == 'test']

print(f'Train+dev : {len(train_meta)}')
print(f'Test      : {len(test_meta)}')
print('\nTrain label distribution:')
dist = Counter(m['label'] for m in train_meta)
for c in range(4):
    print(f'  Class {c} ({CLASS_NAMES[c]}): {dist[c]}')

Train+dev : 110204
Test      : 26100

Train label distribution:
  Class 0 (Real/Real): 29525
  Class 1 (FakeVideo/RealAudio): 27091
  Class 2 (RealVideo/FakeAudio): 26797
  Class 3 (Fake/Fake): 26791


In [10]:
# ── CELL 5: EXTRACTION FUNCTIONS ──────────────────────────────────────────

def detect_face_box(frame):
    try:
        faces = RetinaFace.detect_faces(frame)
        if isinstance(faces, dict):
            face = list(faces.values())[0]
            return face['facial_area']
    except:
        pass
    return None


def crop_face(frame, face_box):
    """Crop face region or fall back to center crop if no face detected."""
    h, w, _ = frame.shape
    if face_box is not None:
        x1, y1, x2, y2 = face_box
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        crop = frame[y1:y2, x1:x2]
    else:
        size = min(h, w) // 2
        cx, cy = w // 2, h // 2
        crop = frame[cy-size:cy+size, cx-size:cx+size]
    return cv2.resize(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB),
        (FRAME_SIZE, FRAME_SIZE)
    )


def detect_face_box_from_video(video_path, max_frames=30):
    """Detect face box from the first max_frames frames of a video."""
    cap = cv2.VideoCapture(video_path)
    face_box = None
    for _ in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        face_box = detect_face_box(frame)
        if face_box is not None:
            break
    cap.release()
    return face_box


def extract_frames_uniform(video_path, n=NUM_FRAMES):
    face_box = detect_face_box_from_video(video_path)
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release(); return []
    indices = np.linspace(0, total-1, n, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret: continue
        frames.append(crop_face(frame, face_box))
    cap.release()
    return frames


def extract_frames_from_periods(video_path, fake_periods, n=NUM_FRAMES):
    face_box = detect_face_box_from_video(video_path)
    cap   = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or total < 1:
        cap.release()
        return extract_frames_uniform(video_path, n)

    sorted_periods = sorted(fake_periods, key=lambda x: x[0])
    valid = []
    for start_s, end_s in sorted_periods:
        start_f = max(0, int(start_s * fps))
        end_f   = min(total - 1, int(end_s * fps))
        if end_f > start_f:
            valid.extend(range(start_f, end_f + 1))

    if len(valid) < MIN_FAKE_FRAMES:
        cap.release()
        return extract_frames_uniform(video_path, n)

    chosen = [valid[i] for i in np.linspace(0, len(valid)-1, n, dtype=int)]
    frames = []
    for idx in chosen:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret: continue
        frames.append(crop_face(frame, face_box))
    cap.release()
    return frames


def load_audio(video_path):
    tmp = '/tmp/lav_audio.wav'
    subprocess.run(
        ['ffmpeg', '-y', '-i', video_path, '-ac', '1', '-ar', str(SR), tmp],
        capture_output=True
    )
    return librosa.load(tmp, sr=SR)


def mel_segment(segment, sr):
    mel = librosa.feature.melspectrogram(y=segment, sr=sr, n_mels=N_MELS)
    mel = librosa.power_to_db(mel)
    if mel.shape[1] < TARGET_MEL_T:
        mel = np.pad(mel, ((0,0),(0, TARGET_MEL_T - mel.shape[1])))
    else:
        mel = mel[:, :TARGET_MEL_T]
    return mel


def extract_audio_uniform(audio, sr, n=NUM_AUDIO_SEG):
    if len(audio) < n: return []
    seg_len = len(audio) // n
    return [mel_segment(audio[i*seg_len:(i+1)*seg_len], sr) for i in range(n)]


def extract_audio_from_periods(audio, sr, fake_periods, n=NUM_AUDIO_SEG):
    sorted_periods = sorted(fake_periods, key=lambda x: x[0])
    fake_audio = np.array([], dtype=np.float32)
    for start_s, end_s in sorted_periods:
        s = max(0, int(start_s * sr))
        e = min(len(audio), int(end_s * sr))
        if e > s:
            fake_audio = np.concatenate([fake_audio, audio[s:e]])
    if len(fake_audio) < 512:
        return extract_audio_uniform(audio, sr, n)
    seg_len = max(1, len(fake_audio) // n)
    feats   = []
    for i in range(n):
        s   = i * seg_len
        e   = s + seg_len
        seg = fake_audio[s:e] if e <= len(fake_audio) else \
              np.pad(fake_audio[s:], (0, e - len(fake_audio)))
        feats.append(mel_segment(seg, sr))
    return feats


print('Extraction functions ready (with RetinaFace face detection).')

Extraction functions ready (with RetinaFace face detection).


In [11]:
# ── CELL 6: SANITY CHECK ──────────────────────────────────────────────────

def process_entry(m):
    video_path   = os.path.join(LAV_ROOT, m['file'])
    fake_periods = m.get('fake_periods', [])
    has_fake     = len(fake_periods) > 0
    label        = m['label']
    try:
        frames = extract_frames_from_periods(video_path, fake_periods) \
                 if has_fake else extract_frames_uniform(video_path)
        if len(frames) < 8: return None
        while len(frames) < NUM_FRAMES: frames.append(frames[-1])
        frames = frames[:NUM_FRAMES]

        audio_raw, sr = load_audio(video_path)
        audio_feats   = extract_audio_from_periods(audio_raw, sr, fake_periods) \
                        if has_fake else extract_audio_uniform(audio_raw, sr)
        if len(audio_feats) < 8: return None
        while len(audio_feats) < NUM_AUDIO_SEG: audio_feats.append(audio_feats[-1])
        audio_feats = audio_feats[:NUM_AUDIO_SEG]

        return (
            np.array(frames,      dtype=np.uint8),
            np.array(audio_feats, dtype=np.float32),
            label
        )
    except:
        return None


print('Testing on one real entry...')
real_m = next(m for m in train_meta if m['label'] == 0)
r = process_entry(real_m)
print(f'  frames={r[0].shape}  audio={r[1].shape}  label={r[2]}' if r else '  FAILED')

print('Testing on one class-3 (Fake/Fake) entry...')
fake_m = next(m for m in train_meta if m['label'] == 3)
r = process_entry(fake_m)
if r:
    print(f'  frames={r[0].shape}  audio={r[1].shape}  label={r[2]}')
    print(f'  fake_periods={fake_m["fake_periods"]}')
else:
    print('  FAILED')

Testing on one real entry...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5


26-03-09 09:08:30 - Directory /root/.deepface created
26-03-09 09:08:30 - Directory /root/.deepface/weights created
26-03-09 09:08:30 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


100%|██████████| 119M/119M [00:00<00:00, 194MB/s]


  frames=(16, 224, 224, 3)  audio=(16, 128, 32)  label=0
Testing on one class-3 (Fake/Fake) entry...
  frames=(16, 224, 224, 3)  audio=(16, 128, 32)  label=3
  fake_periods=[[4.6, 5.504]]


In [12]:
# ── CELL 7: PREPROCESSING LOOP ────────────────────────────────────────────

def run_preprocessing(meta_list, out_dir, per_class, prefix='lav'):
    done  = set(f for f in os.listdir(out_dir) if f.endswith('.npz'))
    saved = Counter(int(f.split('_')[1]) for f in done)
    print(f'Already saved: {sum(saved.values())} files')
    for c in range(4):
        print(f'  Class {c}: {saved[c]}/{per_class}')

    by_class = {c: [] for c in range(4)}
    for m in meta_list:
        by_class[m['label']].append(m)
    for c in by_class:
        random.shuffle(by_class[c])

    total_new = 0
    for cls in range(4):
        needed = per_class - saved[cls]
        if needed <= 0:
            print(f'\nClass {cls}: complete, skipping.')
            continue
        print(f'\nClass {cls} ({CLASS_NAMES[cls]}) — need {needed} more...')
        n_saved = saved[cls]
        pbar    = tqdm(by_class[cls], desc=f'Class {cls}')
        for m in pbar:
            if n_saved >= per_class: break
            vid_id   = m['file'].replace('/', '_').replace('.mp4', '')
            out_name = f'{prefix}_{cls}_{vid_id}.npz'
            if out_name in done:
                n_saved += 1; continue
            result = process_entry(m)
            if result is None: continue
            frames, audio, label = result
            np.savez_compressed(
                os.path.join(out_dir, out_name),
                frames=frames, audio=audio,
                label=np.array(label)
            )
            n_saved  += 1
            total_new += 1
            pbar.set_postfix({'saved': f'{n_saved}/{per_class}'})
        print(f'  Done: {n_saved}/{per_class}')

    print(f'\nTotal new files this run: {total_new}')

print('Loop ready.')

Loop ready.


In [13]:
# ── CELL 8: PREPROCESS TRAIN ──────────────────────────────────────────────
print('=== TRAIN+DEV ===')
run_preprocessing(train_meta, OUT_TRAIN, TRAIN_PER_CLASS, prefix='lav')

=== TRAIN+DEV ===
Already saved: 0 files
  Class 0: 0/200
  Class 1: 0/200
  Class 2: 0/200
  Class 3: 0/200

Class 0 (Real/Real) — need 200 more...


Class 0:   1%|          | 200/29525 [27:56<68:18:05,  8.38s/it, saved=200/200]


  Done: 200/200

Class 1 (FakeVideo/RealAudio) — need 200 more...


Class 1:   1%|          | 200/27091 [25:46<57:45:03,  7.73s/it, saved=200/200]


  Done: 200/200

Class 2 (RealVideo/FakeAudio) — need 200 more...


Class 2:   1%|          | 200/26797 [25:50<57:16:24,  7.75s/it, saved=200/200]


  Done: 200/200

Class 3 (Fake/Fake) — need 200 more...


Class 3:   1%|          | 200/26791 [26:18<58:18:26,  7.89s/it, saved=200/200]

  Done: 200/200

Total new files this run: 800


In [14]:
# ── CELL 9: PREPROCESS TEST ───────────────────────────────────────────────
print('=== TEST ===')
run_preprocessing(test_meta, OUT_TEST, TEST_PER_CLASS, prefix='lav')

=== TEST ===
Already saved: 0 files
  Class 0: 0/80
  Class 1: 0/80
  Class 2: 0/80
  Class 3: 0/80

Class 0 (Real/Real) — need 80 more...


Class 0:   1%|          | 80/6906 [10:08<14:25:05,  7.60s/it, saved=80/80]


  Done: 80/80

Class 1 (FakeVideo/RealAudio) — need 80 more...


Class 1:   1%|          | 80/6452 [10:06<13:25:46,  7.59s/it, saved=80/80]


  Done: 80/80

Class 2 (RealVideo/FakeAudio) — need 80 more...


Class 2:   1%|▏         | 80/6373 [12:34<16:28:45,  9.43s/it, saved=80/80]


  Done: 80/80

Class 3 (Fake/Fake) — need 80 more...


Class 3:   1%|▏         | 80/6369 [10:12<13:22:27,  7.66s/it, saved=80/80]

  Done: 80/80

Total new files this run: 320


In [15]:
# ── CELL 10: VERIFY ───────────────────────────────────────────────────────
print('=== VERIFICATION ===')
for split_name, out_dir in [('TRAIN', OUT_TRAIN), ('TEST', OUT_TEST)]:
    files  = [f for f in os.listdir(out_dir) if f.endswith('.npz')]
    labels = [int(f.split('_')[1]) for f in files]
    print(f'\n{split_name}: {len(files)} total')
    for c, cnt in sorted(Counter(labels).items()):
        print(f'  Class {c} ({CLASS_NAMES[c]}): {cnt}')

sample_f = os.path.join(OUT_TRAIN, os.listdir(OUT_TRAIN)[0])
d = np.load(sample_f)
print(f'\nSample: {os.path.basename(sample_f)}')
print(f'  frames : {d["frames"].shape}  (expect (16,224,224,3))')
print(f'  audio  : {d["audio"].shape}   (expect (16,128,32))')
print(f'  label  : {d["label"]}')

=== VERIFICATION ===

TRAIN: 800 total
  Class 0 (Real/Real): 200
  Class 1 (FakeVideo/RealAudio): 200
  Class 2 (RealVideo/FakeAudio): 200
  Class 3 (Fake/Fake): 200

TEST: 320 total
  Class 0 (Real/Real): 80
  Class 1 (FakeVideo/RealAudio): 80
  Class 2 (RealVideo/FakeAudio): 80
  Class 3 (Fake/Fake): 80

Sample: lav_0_train_124828.npz
  frames : (16, 224, 224, 3)  (expect (16,224,224,3))
  audio  : (16, 128, 32)   (expect (16,128,32))
  label  : 0
